# SIFLOW · run_0 · Smoke test

Verifies the install, the unit-test suite (incl. the SUBS check), and an end-to-end MDLM load + one-step generation. Run this first.

**Runtime:** comfortably under one Colab session.

**How to use:** run every cell top-to-bottom. Cells 1–2 set up the environment and the artifact location; the last cell downloads this part's output zip for the next notebook.

In [ ]:
# === 1. Clone + install (run once per session, ~2 min) ===
REPO_URL = "https://github.com/kagtgi/siflow.git"
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# === 2. Where do artifacts live? ===
# Default: a local folder + zip download/upload between parts (no Drive needed).
# Set USE_DRIVE = True to persist on Google Drive instead (then the import and
# download steps below become no-ops).
USE_DRIVE = False

import os
if USE_DRIVE:
    from siflow.utils import drive
    drive.mount()
    os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
    BASE = drive.base_dir()
else:
    BASE = "/content/artifacts"
    os.makedirs(BASE, exist_ok=True)
print("artifacts ->", BASE)

In [ ]:
!python -m pytest tests/ -q

In [ ]:
import torch
from siflow.teacher import MDLMTeacher
from siflow.head import VelocityHead
from siflow.student import Student
from transformers import AutoTokenizer

teacher = MDLMTeacher(dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained("gpt2")
ids = torch.full((2, 32), teacher.mask_index, device=teacher.device)
z, _ = teacher.logits_and_hidden(ids)
print("max prob on mask token (should be ~0):",
      torch.softmax(z.float(), -1)[..., teacher.mask_index].max().item())
head = VelocityHead(teacher.hidden_dim, teacher.embedding_matrix, bottleneck=1024).to(teacher.device)
print(tok.decode(Student(teacher, head).generate(2, 32, k=1)[0].tolist()))
print("smoke OK")